In [36]:
print("Test")

Test


In [37]:
from pathlib import Path
import numpy as np  
import pandas as pd

#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

Data folder is ready: C:\Users\User\Documents\GitHub\AI-and-ML-Laboratory\data


## First attempt to get information from one URL

In [38]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://www.rottentomatoes.com/m/i_love_boosters"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

resp = requests.get(url, headers=headers, timeout=10)

if resp.status_code != 200:
    print("Error HTTP:", resp.status_code)
else:
    soup = BeautifulSoup(resp.content, "html.parser")

    title = None
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

    print(f"Título: {title}")

    tomatometer = None
    popcornmeter = None

    score_board = soup.find("score-board") or soup.find("scoreboard") or soup.find("scoreBoard")

    if score_board:
        tomatometer = (
            score_board.get("tomatometerscore")
            or score_board.get("tomatometer")
        )

        popcornmeter = (
            score_board.get("popcornmeterscore")
            or score_board.get("popcornmeter")
            or score_board.get("audiencescore")
            or score_board.get("audienceScore")
        )

    # Fallback por texto visible: "63% Tomatometer ... 89% Popcornmeter"
    page_text = soup.get_text(" ", strip=True)

    if tomatometer is None:
        match = re.search(r"(\d+)%\s+Tomatometer", page_text, re.I)
        if match:
            tomatometer = match.group(1) + "%"

    if popcornmeter is None:
        match = re.search(r"(\d+)%\s+Popcornmeter", page_text, re.I)
        if match:
            popcornmeter = match.group(1) + "%"

    print(f"Tomatometer: {tomatometer}")
    print(f"Popcornmeter: {popcornmeter}")

Título: I Love Boosters
Tomatometer: 92%
Popcornmeter: 75%


## Second attempt to get complete information from one URL

In [39]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd

url = "https://www.rottentomatoes.com/m/star_wars_the_mandalorian_and_grogu"

# url = "https://www.rottentomatoes.com/m/mile_end_kicks"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
resp = requests.get(url, headers=headers, timeout=10)
if resp.status_code != 200:
    print("Error HTTP:", resp.status_code)
else:
    soup = BeautifulSoup(resp.content, "html.parser")


###################### Título (fallbacks) ################################
    title = None
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]
    print(f"Título: {title}")  


######################## Critics And summary ########################

script = soup.find("script", id="media-scorecard-json")

if script and script.string:
    data = json.loads(script.string)

    summary = data.get("description")

    tomatometer = data.get("criticsScore", {}).get("scorePercent")
    popcornmeter = data.get("audienceScore", {}).get("scorePercent")

    print("Resumen:", summary)
    print("Tomatometer:", tomatometer)
    print("Popcornmeter:", popcornmeter)


################## Platform Names ######################################
page_text = soup.get_text(" ", strip=True)

platform_names = []

for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
    a = li.find("a", href=True)

    if a and "affiliates:" in a["href"]:
        name = a.get_text(strip=True)
        platform_names.append(name)

platform_names = list(dict.fromkeys(platform_names))

print(platform_names)


####################### Critics consensus #############################

critics_consensus = None

# Opción 1: buscar por id
consensus_div = soup.find("div", id="critics-consensus")

if consensus_div:
    p = consensus_div.find("p")
    if p:
        critics_consensus = p.get_text(" ", strip=True)



print("Critics Consensus:", critics_consensus, '\n')

################# Dictionary of each movie to create my Dataset
rows = []

movie_row = {
    "title": title,
    # "url": url, 
    "tomatometer": tomatometer,
    "popcornmeter": popcornmeter,
    "summary": summary,
    "critics_consensus": critics_consensus,
    "platforms": ", ".join(platform_names)
}

#########Creating the dataframe
print(movie_row)
rows.append(movie_row)
df = pd.DataFrame(rows)
df


Título: Star Wars: The Mandalorian and Grogu
Resumen: The evil Empire has fallen, and Imperial warlords remain scattered throughout the galaxy. As the fledgling New Republic works to protect everything the Rebellion fought for, they have enlisted the help of legendary Mandalorian bounty hunter Din Djarin (Pedro Pascal) and his young apprentice Grogu.
Tomatometer: 62%
Popcornmeter: 89%
['Fandango at Home', 'Netflix', 'Apple TV', 'Prime Video']
Critics Consensus: Bountiful in action but threadbare in narrative thrust with its episodic structure, this Star Wars is more of a skirmish that coasts on the charm of its central dynamic duo. 

{'title': 'Star Wars: The Mandalorian and Grogu', 'tomatometer': '62%', 'popcornmeter': '89%', 'summary': 'The evil Empire has fallen, and Imperial warlords remain scattered throughout the galaxy. As the fledgling New Republic works to protect everything the Rebellion fought for, they have enlisted the help of legendary Mandalorian bounty hunter Din Djarin

,title,tomatometer,popcornmeter,summary,critics_consensus,platforms
0,Star Wars: The Mandalorian and Grogu,62%,89%,"The evil Empire has fallen, and Imperial warlo...",Bountiful in action but threadbare in narrativ...,"Fandango at Home, Netflix, Apple TV, Prime Video"


In [40]:
## Check the values per page to check how many pages will be scrolled
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1,10):  # 1 hasta 10
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    page_movies = []

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            page_movies.append(full_url)

    page_movies = list(dict.fromkeys(page_movies))
    print(f"Series encontradas en página {page}: {len(page_movies)}")

    all_movies_urls.extend(page_movies)

    time.sleep(1)

all_movies_urls = list(dict.fromkeys(all_movies_urls))

##### Summarize of all movies founded
print(f"\nTotal películas únicas: {len(all_movies_urls)}")
for movie in all_movies_urls:
    print(movie)
# At the end there are Just 154 unic values, so it could be scrolled until page 5

Series encontradas en página 1: 42
Series encontradas en página 2: 70
Series encontradas en página 3: 98
Series encontradas en página 4: 126
Error en página 5: HTTP 404
Error en página 6: HTTP 404
Error en página 7: HTTP 404
Error en página 8: HTTP 404
Error en página 9: HTTP 404

Total películas únicas: 126
https://www.rottentomatoes.com/tv/the_boroughs/s01
https://www.rottentomatoes.com/tv/maximum_pleasure_guaranteed/s01
https://www.rottentomatoes.com/tv/mating_season/s01
https://www.rottentomatoes.com/tv/kylie/s01
https://www.rottentomatoes.com/tv/youre_killing_me/s01
https://www.rottentomatoes.com/tv/skymed/s04
https://www.rottentomatoes.com/tv/the_chi/s08
https://www.rottentomatoes.com/tv/the_boys_2019/s05
https://www.rottentomatoes.com/tv/spider_noir/s01
https://www.rottentomatoes.com/tv/off_campus/s01
https://www.rottentomatoes.com/tv/widows_bay/s01
https://www.rottentomatoes.com/tv/legends_2026/s01
https://www.rottentomatoes.com/tv/nemesis_2026/s01
https://www.rottentomatoes.co

In [41]:
# Path for files pkl wich are smaller than JSON files
# This File will contain the dataset of Series
pickle_path = DATA_DIR / "series_urls.pkl"

In [42]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_series_urls = []

for page in range(1, 5):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            all_series_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_series_urls = list(dict.fromkeys(all_series_urls))

print(f"\nTotal de series encontradas: {len(all_series_urls)}")

for u in all_series_urls:
    print(u)


with open(pickle_path, "wb") as f:
    pickle.dump(all_series_urls, f)




Total de series encontradas: 126
https://www.rottentomatoes.com/tv/the_boroughs/s01
https://www.rottentomatoes.com/tv/maximum_pleasure_guaranteed/s01
https://www.rottentomatoes.com/tv/mating_season/s01
https://www.rottentomatoes.com/tv/kylie/s01
https://www.rottentomatoes.com/tv/youre_killing_me/s01
https://www.rottentomatoes.com/tv/skymed/s04
https://www.rottentomatoes.com/tv/the_chi/s08
https://www.rottentomatoes.com/tv/the_boys_2019/s05
https://www.rottentomatoes.com/tv/spider_noir/s01
https://www.rottentomatoes.com/tv/off_campus/s01
https://www.rottentomatoes.com/tv/widows_bay/s01
https://www.rottentomatoes.com/tv/legends_2026/s01
https://www.rottentomatoes.com/tv/nemesis_2026/s01
https://www.rottentomatoes.com/tv/rivals_2024/s02
https://www.rottentomatoes.com/tv/the_boys_2019
https://www.rottentomatoes.com/tv/off_campus
https://www.rottentomatoes.com/tv/nemesis_2026
https://www.rottentomatoes.com/tv/the_boroughs
https://www.rottentomatoes.com/tv/legends_2026
https://www.rottentom

In [43]:
### Just for verify the data
# df_imported = pd.read_pickle(pickle_path)
#df_imported

#### USER-AGENT REQUEST -----> TO TAKE ALL THE DATA FROM THE URLs

In [44]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd
import time


# "User-Agent" REQUEST simula que la petición como si viniera desde un navegador Chrome en Windows para evitar bloqueos de HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}



###########################    DEFINICIÓN DE MI FUNCIÓN PARA CADA URL    ###########################
def scrape_movie(url):
    resp = requests.get(url, headers=headers, timeout=10) ## Uso de el agente para hacer la solicitud como si viniera del navegador

    if resp.status_code != 200:
        print(f"Error HTTP {resp.status_code}: {url}")
        return None

    soup = BeautifulSoup(resp.content, "html.parser")

    # Título
    title = None
    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

####################### 1 . CALLING GENRE(S) 

    genre = None # create the variable

    script = soup.find("script", id="mps-page-integration")

    if script:
        script_text = script.get_text()

        match = re.search(r'"cag\[genre\]":"([^"]+)"', script_text) #Looking for the text

        if match:
            genre = match.group(1)
    
    genres = [] #Save genres

    if genre:
        genres = genre.split("|") # Spplit them 

    # print("Genre:", genre)
    
################# 2. CALLING: Summary, Tomatometer, Popcornmeter

     # Variables 
    summary = None
    tomatometer = None
    popcornmeter = None

    script = soup.find("script", id="media-scorecard-json")

    if script:
        try:
            data = json.loads(script.get_text(strip=True))
            
            #Checking for texts, different forms to get the values
            summary = data.get("description")
            tomatometer = data.get("criticsScore", {}).get("scorePercent")
            popcornmeter = data.get("audienceScore", {}).get("scorePercent")

        except json.JSONDecodeError:
            pass

 ############### 3. STREAM PLATFORMS 
    platform_names = []

    for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
        a = li.find("a", href=True)

        if a and "affiliates:" in a["href"]:
            name = a.get_text(strip=True)
            platform_names.append(name)

    platform_names = list(dict.fromkeys(platform_names))

############### 4. CRITICS CONSENSUS
    critics_consensus = None

    consensus_div = soup.find("div", id="critics-consensus")

    if consensus_div:
        p = consensus_div.find("p")
        if p:
            critics_consensus = p.get_text(" ", strip=True)

    movie_row = {
        "title": title,
        "genre": genre,
        "url": url,
        "tomatometer": tomatometer,
        "popcornmeter": popcornmeter,
        "summary": summary,
        "critics_consensus": critics_consensus,
        "platforms": ", ".join(platform_names)
    }

    return movie_row ## Me retorna el valor de cada columna nueva


################################# Fin de la función ######################

In [45]:
#Cargo mi archivo pkl ### MAS ADELANTE SE ACTUALIZA CON TODAS LAS SERIES Y PEL[ICULAS]
with pickle_path.open("rb") as f:
    all_series_urls = pickle.load(f)

In [46]:
#####  For para crear la matriz
rows = []

for i, url in enumerate(all_series_urls, start=1):
    # print(f"Scrapeando {i}/{len(movie_urls)}: {url}")

    movie_row = scrape_movie(url)

    if movie_row is not None:
        rows.append(movie_row)

    time.sleep(1)


#### Creación del DataFrame 
df = pd.DataFrame(rows)



#### Guardar EL dataframe
df.to_csv("rottentomatoes_series_dataset.csv", index=False, encoding="utf-8")

df.head()

,title,genre,url,tomatometer,popcornmeter,summary,critics_consensus,platforms
0,Season 1 – The Boroughs,Drama|Adventure|Mystery & Thriller|Sci-Fi,https://www.rottentomatoes.com/tv/the_boroughs...,95%,85%,In a seemingly picturesque retirement communit...,NaN,"Fandango at Home, Netflix, Apple TV, Prime Video"
1,Season 1 – Maximum Pleasure Guaranteed,Comedy,https://www.rottentomatoes.com/tv/maximum_plea...,92%,86%,Newly divorced mom Paula falls down a rabbit h...,NaN,"Fandango at Home, Netflix, Apple TV, Prime Video"
2,Season 1 – Mating Season,Comedy|Romance|Animation,https://www.rottentomatoes.com/tv/mating_seaso...,75%,NaN,Bring out the animal within to unleash your na...,NaN,"Fandango at Home, Netflix, Apple TV, Prime Video"
3,Season 1 – Kylie,Documentary|Biography,https://www.rottentomatoes.com/tv/kylie/s01,100%,98%,Kylie Minogue opens her personal archives and ...,NaN,"Fandango at Home, Netflix, Apple TV, Prime Video"
4,Season 1 – You're Killing Me,Drama,https://www.rottentomatoes.com/tv/youre_killin...,NaN,NaN,,NaN,"Fandango at Home, Netflix, Apple TV, Prime Video"
